# NOTE:  THIS NOTEBOOK WILL TAKE A 30 MINUTES TO COMPLETE.

# PLEASE BE PATIENT.

# Optimize Models using Automatic Model Tuning

<img src="img/hpt.png" width="90%" align="left">

In [1]:
import boto3
import sagemaker
import pandas as pd

sess = sagemaker.Session()
bucket = sess.default_bucket()
role = sagemaker.get_execution_role()
region = boto3.Session().region_name

sm = boto3.Session().client(service_name="sagemaker", region_name=region)

sagemaker.config INFO - Not applying SDK defaults from location: /etc/xdg/sagemaker/config.yaml
sagemaker.config INFO - Not applying SDK defaults from location: /home/sagemaker-user/.config/sagemaker/config.yaml


# PRE-REQUISITE: _You need to have succesfully run the notebooks in the `PREPARE`section before you continue with this notebook._ 

# Specify the S3 Location of the Features

In [2]:
%store -r processed_train_data_s3_uri

In [3]:
try:
    processed_train_data_s3_uri
    print("[OK]")
except NameError:
    print("+++++++++++++++++++++++++++++++")
    print("[ERROR] Please run the notebooks in the PREPARE section before you continue.")
    print("+++++++++++++++++++++++++++++++")

[OK]


In [4]:
print(processed_train_data_s3_uri)

s3://sagemaker-us-east-1-891377026966/sagemaker-scikit-learn-2024-07-01-22-42-17-522/output/bert-train


In [5]:
%store -r processed_validation_data_s3_uri

In [6]:
try:
    processed_validation_data_s3_uri
    print("[OK]")
except NameError:
    print("+++++++++++++++++++++++++++++++")
    print("[ERROR] Please run the notebooks in the previous sections before you continue.")
    print("+++++++++++++++++++++++++++++++")

[OK]


In [7]:
print(processed_validation_data_s3_uri)

s3://sagemaker-us-east-1-891377026966/sagemaker-scikit-learn-2024-07-01-22-42-17-522/output/bert-validation


In [8]:
%store -r processed_test_data_s3_uri

In [9]:
try:
    processed_test_data_s3_uri
    print("[OK]")
except NameError:
    print("+++++++++++++++++++++++++++++++")
    print("[ERROR] Please run the notebooks in the previous sections before you continue.")
    print("+++++++++++++++++++++++++++++++")

[OK]


In [10]:
print(processed_test_data_s3_uri)

s3://sagemaker-us-east-1-891377026966/sagemaker-scikit-learn-2024-07-01-22-42-17-522/output/bert-test


In [11]:
print(processed_train_data_s3_uri)
!aws s3 ls $processed_train_data_s3_uri/

s3://sagemaker-us-east-1-891377026966/sagemaker-scikit-learn-2024-07-01-22-42-17-522/output/bert-train
2024-07-01 23:00:11   10472476 part-algo-1-amazon_reviews_us_Digital_Software_v1_00.tfrecord
2024-07-01 23:00:11    2313178 part-algo-1-amazon_reviews_us_Gift_Card_v1_00.tfrecord
2024-07-01 23:00:36   11711096 part-algo-2-amazon_reviews_us_Digital_Video_Games_v1_00.tfrecord


In [12]:
print(processed_validation_data_s3_uri)
!aws s3 ls $processed_validation_data_s3_uri/

s3://sagemaker-us-east-1-891377026966/sagemaker-scikit-learn-2024-07-01-22-42-17-522/output/bert-validation
2024-07-01 23:00:11     582429 part-algo-1-amazon_reviews_us_Digital_Software_v1_00.tfrecord
2024-07-01 23:00:11     128035 part-algo-1-amazon_reviews_us_Gift_Card_v1_00.tfrecord
2024-07-01 23:00:36     651681 part-algo-2-amazon_reviews_us_Digital_Video_Games_v1_00.tfrecord


In [13]:
print(processed_test_data_s3_uri)
!aws s3 ls $processed_test_data_s3_uri/

s3://sagemaker-us-east-1-891377026966/sagemaker-scikit-learn-2024-07-01-22-42-17-522/output/bert-test
2024-07-01 23:00:11     582260 part-algo-1-amazon_reviews_us_Digital_Software_v1_00.tfrecord
2024-07-01 23:00:11     128809 part-algo-1-amazon_reviews_us_Gift_Card_v1_00.tfrecord
2024-07-01 23:00:37     650896 part-algo-2-amazon_reviews_us_Digital_Video_Games_v1_00.tfrecord


In [14]:
!pip list

Package                               Version
------------------------------------- ----------------
absl-py                               2.1.0
accelerate                            0.21.0
aiobotocore                           2.12.2
aiohttp                               3.9.5
aioitertools                          0.11.0
aiosignal                             1.3.1
aiosqlite                             0.19.0
altair                                5.3.0
amazon-codewhisperer-jupyterlab-ext   2.0.2
amazon_sagemaker_jupyter_scheduler    3.0.11
amazon-sagemaker-sql-editor           0.1.7
amazon-sagemaker-sql-execution        0.1.5
amazon-sagemaker-sql-magic            0.1.1
annotated-types                       0.6.0
ansi2html                             0.0.0
ansicolors                            1.1.8
antlr4-python3-runtime                4.9.3
anyio                                 4.3.0
archspec                              0.2.3
argon2-cffi                           23.1.0
argon2-cffi-b

In [15]:
from sagemaker.inputs import TrainingInput

s3_input_train_data = TrainingInput(s3_data=processed_train_data_s3_uri, distribution="ShardedByS3Key")
s3_input_validation_data = TrainingInput(s3_data=processed_validation_data_s3_uri, distribution="ShardedByS3Key")
s3_input_test_data = TrainingInput(s3_data=processed_test_data_s3_uri, distribution="ShardedByS3Key")

print(s3_input_train_data.config)
print(s3_input_validation_data.config)
print(s3_input_test_data.config)

{'DataSource': {'S3DataSource': {'S3DataType': 'S3Prefix', 'S3Uri': 's3://sagemaker-us-east-1-891377026966/sagemaker-scikit-learn-2024-07-01-22-42-17-522/output/bert-train', 'S3DataDistributionType': 'ShardedByS3Key'}}}
{'DataSource': {'S3DataSource': {'S3DataType': 'S3Prefix', 'S3Uri': 's3://sagemaker-us-east-1-891377026966/sagemaker-scikit-learn-2024-07-01-22-42-17-522/output/bert-validation', 'S3DataDistributionType': 'ShardedByS3Key'}}}
{'DataSource': {'S3DataSource': {'S3DataType': 'S3Prefix', 'S3Uri': 's3://sagemaker-us-east-1-891377026966/sagemaker-scikit-learn-2024-07-01-22-42-17-522/output/bert-test', 'S3DataDistributionType': 'ShardedByS3Key'}}}


In [16]:
!cat src/tf_bert_reviews.py

import time
import random
import pandas as pd
from glob import glob
import pprint
import argparse
import json
import subprocess
import sys
import os
import csv

# subprocess.check_call([sys.executable, '-m', 'pip', 'install', 'tensorflow==2.1.0'])
import tensorflow as tf
import pandas as pd
import numpy as np

subprocess.check_call([sys.executable, "-m", "pip", "install", "transformers==3.5.1"])
# subprocess.check_call([sys.executable, '-m', 'pip', 'install', 'sagemaker-tensorflow==2.1.0.1.0.0'])
# subprocess.check_call([sys.executable, '-m', 'pip', 'install', 'smdebug==0.9.3'])
subprocess.check_call([sys.executable, "-m", "pip", "install", "scikit-learn==0.23.1"])
subprocess.check_call([sys.executable, "-m", "pip", "install", "matplotlib==3.2.1"])

from transformers import DistilBertTokenizer
from transformers import DistilBertConfig
from transformers import TFDistilBertForSequenceClassification

from tensorflow.keras.callbacks import ModelCheckpoint
from tensorflow.keras.models impor

# Setup Static Hyper-Parameters for Classification Layer
First, retrieve `max_seq_length` from the prepare phase.

In [17]:
%store -r max_seq_length

In [18]:
try:
    max_seq_length
    print("[OK]")
except NameError:
    print("+++++++++++++++++++++++++++++++")
    print("[ERROR] Please run the notebooks in the PREPARE section before you continue.")
    print("+++++++++++++++++++++++++++++++")

[OK]


In [19]:
print(max_seq_length)

64


In [20]:
epochs = 3
epsilon = 0.00000001
validation_batch_size = 128
test_batch_size = 128
train_steps_per_epoch = 100
validation_steps = 100
test_steps = 100
train_instance_count = 1
train_instance_type = "ml.m5.xlarge"  # evt
# train_instance_type='ml.m5.4xlarge' #bur
train_volume_size = 1024
use_xla = True
use_amp = True
enable_sagemaker_debugger = False
enable_checkpointing = False
enable_tensorboard = False
input_mode = "File"
run_validation = True
run_test = True
run_sample_predictions = True

# Track the Optimizations Within our Experiment

In [21]:
%store -r experiment_name

In [22]:
try:
    experiment_name
    print("[OK]")
except NameError:
    print("+++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++")
    print("[ERROR] Please run the notebooks in the TRAIN section before you continue.")
    print("+++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++")

[OK]


In [23]:
print(experiment_name)

Amazon-Customer-Reviews-BERT-Experiment-1719873607


In [24]:
%store -r trial_name

In [25]:
try:
    trial_name
    print("[OK]")
except NameError:
    print("+++++++++++++++++++++++++++++++")
    print("[ERROR] Please run the notebooks in the previous TRAIN section before you continue.")
    print("+++++++++++++++++++++++++++++++")

[OK]


In [26]:
print(trial_name)

trial-1719873613


In [28]:
!pip install sagemaker-experiments

  Using cached sagemaker_experiments-0.1.45-py3-none-any.whl.metadata (10 kB)
Using cached sagemaker_experiments-0.1.45-py3-none-any.whl (42 kB)


In [29]:
import time
from smexperiments.trial import Trial

timestamp = "{}".format(int(time.time()))

trial = Trial.load(trial_name=trial_name)
print(trial)

Trial(sagemaker_boto_client=<botocore.client.SageMaker object at 0x7fa277d69450>,trial_name='trial-1719873613',trial_arn='arn:aws:sagemaker:us-east-1:891377026966:experiment-trial/trial-1719873613',display_name='trial-1719873613',experiment_name='Amazon-Customer-Reviews-BERT-Experiment-1719873607',creation_time=datetime.datetime(2024, 7, 1, 22, 40, 13, 898000, tzinfo=tzlocal()),created_by={'UserProfileArn': 'arn:aws:sagemaker:us-east-1:891377026966:user-profile/d-fu4acyc03cpg/default-20240628T190970', 'UserProfileName': 'default-20240628T190970', 'DomainId': 'd-fu4acyc03cpg'},last_modified_time=datetime.datetime(2024, 7, 2, 18, 58, 1, 719000, tzinfo=tzlocal()),last_modified_by={},response_metadata={'RequestId': '3624b7fd-bb2b-4b94-9b16-b66a23a21966', 'HTTPStatusCode': 200, 'HTTPHeaders': {'x-amzn-requestid': '3624b7fd-bb2b-4b94-9b16-b66a23a21966', 'content-type': 'application/x-amz-json-1.1', 'content-length': '509', 'date': 'Tue, 02 Jul 2024 22:44:21 GMT'}, 'RetryAttempts': 0})


In [30]:
from smexperiments.tracker import Tracker

tracker_optimize = Tracker.create(display_name="optimize-1", sagemaker_boto_client=sm)

optimize_trial_component_name = tracker_optimize.trial_component.trial_component_name
print("Optimize trial component name {}".format(optimize_trial_component_name))

Optimize trial component name TrialComponent-2024-07-02-224426693344-vnzt


# Attach the `deploy` Trial Component and Tracker as a Component to the Trial

In [31]:
trial.add_trial_component(tracker_optimize.trial_component)

# Setup Dynamic Hyper-Parameter Ranges to Explore


In [32]:
from sagemaker.tuner import IntegerParameter
from sagemaker.tuner import ContinuousParameter
from sagemaker.tuner import CategoricalParameter
from sagemaker.tuner import HyperparameterTuner

hyperparameter_ranges = {
    "learning_rate": ContinuousParameter(0.00001, 0.00005, scaling_type="Linear"),
    "train_batch_size": CategoricalParameter([128, 256]),
    "freeze_bert_layer": CategoricalParameter([True, False]),
}

# Setup Metrics

In [33]:
metrics_definitions = [
    {"Name": "train:loss", "Regex": "loss: ([0-9\\.]+)"},
    {"Name": "train:accuracy", "Regex": "accuracy: ([0-9\\.]+)"},
    {"Name": "validation:loss", "Regex": "val_loss: ([0-9\\.]+)"},
    {"Name": "validation:accuracy", "Regex": "val_accuracy: ([0-9\\.]+)"},
]

In [34]:
from sagemaker.tensorflow import TensorFlow

estimator = TensorFlow(
    entry_point="tf_bert_reviews.py",
    source_dir="src",
    role=role,
    instance_count=train_instance_count,
    instance_type=train_instance_type,
    volume_size=train_volume_size,
    py_version="py37",
    framework_version="2.3.1",
    hyperparameters={
        "epochs": epochs,
        "epsilon": epsilon,
        "validation_batch_size": validation_batch_size,
        "test_batch_size": test_batch_size,
        "train_steps_per_epoch": train_steps_per_epoch,
        "validation_steps": validation_steps,
        "test_steps": test_steps,
        "use_xla": use_xla,
        "use_amp": use_amp,
        "max_seq_length": max_seq_length,
        "enable_sagemaker_debugger": enable_sagemaker_debugger,
        "enable_checkpointing": enable_checkpointing,
        "enable_tensorboard": enable_tensorboard,
        "run_validation": run_validation,
        "run_test": run_test,
        "run_sample_predictions": run_sample_predictions,
    },
    input_mode=input_mode,
    metric_definitions=metrics_definitions,
    #                       max_run=7200 # max 2 hours * 60 minutes seconds per hour * 60 seconds per minute
)

# Setup HyperparameterTuner with Estimator and Hyper-Parameter Ranges

In [35]:
objective_metric_name = "train:accuracy"

tuner = HyperparameterTuner(
    estimator=estimator,
    objective_type="Maximize",
    objective_metric_name=objective_metric_name,
    hyperparameter_ranges=hyperparameter_ranges,
    metric_definitions=metrics_definitions,
    max_jobs=2,
    max_parallel_jobs=1,
    strategy="Bayesian",
    early_stopping_type="Auto",
)

# Start Tuning Job

In [36]:
tuner.fit(
    inputs={"train": s3_input_train_data, "validation": s3_input_validation_data, "test": s3_input_test_data},
    include_cls_metadata=False,
    wait=False,
)

INFO:sagemaker.image_uris:image_uri is not presented, retrieving image_uri based on instance_type, framework etc.
INFO:sagemaker.image_uris:image_uri is not presented, retrieving image_uri based on instance_type, framework etc.
INFO:sagemaker.image_uris:image_uri is not presented, retrieving image_uri based on instance_type, framework etc.
INFO:sagemaker:Creating hyperparameter tuning job with name: tensorflow-training-240702-2245


# Check Tuning Job Status
Re-run this cell to track the status.

In [37]:
from pprint import pprint

tuning_job_name = tuner.latest_tuning_job.job_name

In [38]:
from IPython.core.display import display, HTML

display(
    HTML(
        '<b>Review <a target="blank" href="https://console.aws.amazon.com/sagemaker/home?region={}#/hyper-tuning-jobs/{}">Hyper-Parameter Tuning Job</a></b>'.format(
            region, tuning_job_name
        )
    )
)

/tmp/ipykernel_2449/399046009.py:1: DeprecationWarning: Importing display from IPython.core.display is deprecated since IPython 7.14, please import from IPython display
  from IPython.core.display import display, HTML


# _Please Wait for the ^^ Tuning Job ^^ to Complete Above_

In [39]:
%%time

tuner.wait()

........................................................................................................................................................................................................................................................................................................................................................................................................................................................................................................................................................................................................................................................................................................................................................................................................................................................................................................................................................................................................................................

# [INFO] _Feel free to continue to the next workshop section while this notebook is running._

# Show the Tuning Job
### _Note:  This will fail at first.  Please wait about 15-30 seconds and re-run._

In [40]:
from sagemaker.analytics import HyperparameterTuningJobAnalytics

hp_results = HyperparameterTuningJobAnalytics(sagemaker_session=sess, hyperparameter_tuning_job_name=tuning_job_name)

df_results = hp_results.dataframe()
df_results.shape

(2, 9)

In [41]:
df_results.sort_values("FinalObjectiveValue", ascending=0)

,freeze_bert_layer,learning_rate,train_batch_size,TrainingJobName,TrainingJobStatus,FinalObjectiveValue,TrainingStartTime,TrainingEndTime,TrainingElapsedTimeSeconds
1,"""False""",0.000042,"""128""",tensorflow-training-240702-2245-001-0ab8e046,Completed,0.4837,2024-07-02 22:45:53+00:00,2024-07-03 00:14:58+00:00,5345.0
0,"""True""",0.000029,"""128""",tensorflow-training-240702-2245-002-54822820,Stopped,0.3125,2024-07-03 00:15:44+00:00,2024-07-03 00:30:56+00:00,912.0


# Show the Best Candidate

In [42]:
df_results.sort_values("FinalObjectiveValue", ascending=0).head(1)

,freeze_bert_layer,learning_rate,train_batch_size,TrainingJobName,TrainingJobStatus,FinalObjectiveValue,TrainingStartTime,TrainingEndTime,TrainingElapsedTimeSeconds
1,"""False""",0.000042,"""128""",tensorflow-training-240702-2245-001-0ab8e046,Completed,0.4837,2024-07-02 22:45:53+00:00,2024-07-03 00:14:58+00:00,5345.0


# Log the Best Hyper-Parameter and Objective Metric in the Experiment

Logging `learning_rate` parameter and `accuracy` metric

In [43]:
best_learning_rate = df_results.sort_values("FinalObjectiveValue", ascending=0).head(1)["learning_rate"]
print(best_learning_rate)

1    0.000042
Name: learning_rate, dtype: float64


In [44]:
best_accuracy = df_results.sort_values("FinalObjectiveValue", ascending=0).head(1)["FinalObjectiveValue"]
print(best_accuracy)

1    0.4837
Name: FinalObjectiveValue, dtype: float64


In [45]:
tracker_optimize.log_parameters({"learning_rate": float(best_learning_rate)})

# must save after logging
tracker_optimize.trial_component.save()

/tmp/ipykernel_2449/3757020509.py:1: FutureWarning: Calling float on a single element Series is deprecated and will raise a TypeError in the future. Use float(ser.iloc[0]) instead
  tracker_optimize.log_parameters({"learning_rate": float(best_learning_rate)})


TrialComponent(sagemaker_boto_client=<botocore.client.SageMaker object at 0x7fa2798e8b80>,trial_component_name='TrialComponent-2024-07-02-224426693344-vnzt',display_name='optimize-1',tags=None,trial_component_arn='arn:aws:sagemaker:us-east-1:891377026966:experiment-trial-component/TrialComponent-2024-07-02-224426693344-vnzt',response_metadata={'RequestId': 'e74708cf-4249-4ed4-a3cf-c4e5cd57b5f8', 'HTTPStatusCode': 200, 'HTTPHeaders': {'x-amzn-requestid': 'e74708cf-4249-4ed4-a3cf-c4e5cd57b5f8', 'content-type': 'application/x-amz-json-1.1', 'content-length': '135', 'date': 'Wed, 03 Jul 2024 00:50:51 GMT'}, 'RetryAttempts': 0},parameters={'learning_rate': 4.184158274319715e-05},input_artifacts={},output_artifacts={})

In [46]:
tracker_optimize.log_metric("accuracy", float(best_accuracy))

# must save after logging
tracker_optimize.trial_component.save()

/tmp/ipykernel_2449/312790556.py:1: FutureWarning: Calling float on a single element Series is deprecated and will raise a TypeError in the future. Use float(ser.iloc[0]) instead
  tracker_optimize.log_metric("accuracy", float(best_accuracy))


TrialComponent(sagemaker_boto_client=<botocore.client.SageMaker object at 0x7fa2798e8b80>,trial_component_name='TrialComponent-2024-07-02-224426693344-vnzt',display_name='optimize-1',tags=None,trial_component_arn='arn:aws:sagemaker:us-east-1:891377026966:experiment-trial-component/TrialComponent-2024-07-02-224426693344-vnzt',response_metadata={'RequestId': 'bb066e3c-c6f1-4c29-9d34-5e73e7789f76', 'HTTPStatusCode': 200, 'HTTPHeaders': {'x-amzn-requestid': 'bb066e3c-c6f1-4c29-9d34-5e73e7789f76', 'content-type': 'application/x-amz-json-1.1', 'content-length': '135', 'date': 'Wed, 03 Jul 2024 00:50:54 GMT'}, 'RetryAttempts': 0},parameters={'learning_rate': 4.184158274319715e-05},input_artifacts={},output_artifacts={})

# Show Experiment Analytics

In [47]:
from sagemaker.analytics import ExperimentAnalytics

lineage_table = ExperimentAnalytics(
    sagemaker_session=sess,
    experiment_name=experiment_name,
    metric_names=["validation:accuracy"],
    sort_by="CreationTime",
    sort_order="Descending",
)

df_lineage = lineage_table.dataframe()
df_lineage.shape

(4, 71)

In [48]:
df_lineage

,TrialComponentName,DisplayName,learning_rate,Trials,Experiments,SourceArn,SageMaker.InstanceCount,SageMaker.InstanceType,SageMaker.VolumeSizeInGB,SageMaker.ImageUri - MediaType,...,SageMaker.ModelArtifact - Value,AWS_DEFAULT_REGION,raw-input-data - MediaType,raw-input-data - Value,bert-test - MediaType,bert-test - Value,bert-train - MediaType,bert-train - Value,bert-validation - MediaType,bert-validation - Value
0,TrialComponent-2024-07-02-224426693344-vnzt,optimize-1,0.000042,[trial-1719873613],[Amazon-Customer-Reviews-BERT-Experiment-17198...,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,sagemaker-scikit-learn-2024-07-02-18-40-05-240...,evaluate,NaN,[trial-1719873613],[Amazon-Customer-Reviews-BERT-Experiment-17198...,arn:aws:sagemaker:us-east-1:891377026966:proce...,1.0,ml.t3.xlarge,30.0,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,tensorflow-training-2024-07-02-02-08-36-935-aw...,train,0.000010,[trial-1719873613],[Amazon-Customer-Reviews-BERT-Experiment-17198...,arn:aws:sagemaker:us-east-1:891377026966:train...,1.0,ml.c5.4xlarge,1024.0,NaN,...,s3://sagemaker-us-east-1-891377026966/tensorfl...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,sagemaker-scikit-learn-2024-07-01-22-42-17-522...,prepare,NaN,[trial-1719873613],[Amazon-Customer-Reviews-BERT-Experiment-17198...,arn:aws:sagemaker:us-east-1:891377026966:proce...,2.0,ml.t3.xlarge,30.0,NaN,...,NaN,us-east-1,NaN,s3://sagemaker-us-east-1-891377026966/amazon-r...,NaN,s3://sagemaker-us-east-1-891377026966/sagemake...,NaN,s3://sagemaker-us-east-1-891377026966/sagemake...,NaN,s3://sagemaker-us-east-1-891377026966/sagemake...


# Pass `tuning_job_name` to the Next Notebook

In [49]:
print(tuning_job_name)

tensorflow-training-240702-2245


In [50]:
%store tuning_job_name

Stored 'tuning_job_name' (str)


In [51]:
%store

Stored variables and their in-db values:
balance_dataset                                       -> True
balanced_bias_data_jsonlines_s3_uri                   -> 's3://sagemaker-us-east-1-891377026966/bias-detect
balanced_bias_data_s3_uri                             -> 's3://sagemaker-us-east-1-891377026966/bias-detect
bias_data_s3_uri                                      -> 's3://sagemaker-us-east-1-891377026966/bias-detect
experiment_name                                       -> 'Amazon-Customer-Reviews-BERT-Experiment-171987360
feature_group_name                                    -> 'reviews-feature-group-1719873648'
feature_store_offline_prefix                          -> 'reviews-feature-store-1719873648'
ingest_create_athena_db_passed                        -> True
ingest_create_athena_table_parquet_passed             -> True
ingest_create_athena_table_tsv_passed                 -> True
max_seq_length                                        -> 64
processed_metrics_s3_uri           

# Release Resources

In [52]:
%%html

<p><b>Shutting down your kernel for this notebook to release resources.</b></p>
<button class="sm-command-button" data-commandlinker-command="kernelmenu:shutdown" style="display:none;">Shutdown Kernel</button>
        
<script>
try {
    els = document.getElementsByClassName("sm-command-button");
    els[0].click();
}
catch(err) {
    // NoOp
}    
</script>

In [ ]:
%%javascript

try {
    Jupyter.notebook.save_checkpoint();
    Jupyter.notebook.session.delete();
}
catch(err) {
    // NoOp
}